ZeroMQ: Request-Reply-Pattern

Ggfs. Requirements installieren.

In [1]:
!pip install pyzmq

Imports

In [2]:
import zmq
import threading
import time

Den Server definieren: Request-Reply-Pattern.
Also legen wir einen Reply Socket an.

In [3]:
def run_server():
    context = zmq.Context()
    socket = context.socket(zmq.REP)  # Reply-Socket

    socket.bind("tcp://*:12345")
    print("🟢 ZeroMQ Server läuft...")

    while True:
        message = socket.recv_string()  # blocking
        print("📥 Server empfängt:", message)

        if "STOP" in message:
            print("🛑 Server wird beendet")
            break

        reply = message + " *"
        socket.send_string(reply)

    socket.close()
    context.term()

Client und Server können nicht in der gleichen Zelle starten.
Also legen wir für den Server einen eigenen Thread an.

In [4]:
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(1)

🟢 ZeroMQ Server läuft...


Client definieren. Request-Reply-Pattern, also wird der Client einen Request-Socket öffnen.

In [5]:
def run_client():
    context = zmq.Context()
    socket = context.socket(zmq.REQ)  # Request-Socket

    print("🔵 Client verbindet sich...")
    socket.connect("tcp://localhost:12345")

    # Nachricht senden
    msg = "Hello World"
    print("📤 Client sendet:", msg)
    socket.send_string(msg)

    # Antwort empfangen (REQ muss recv nach send machen!)
    reply = socket.recv_string()
    print("📥 Client empfängt:", reply)

    # STOP senden
    socket.send_string("STOP")
    
    socket.close()
    context.term()

Client starten

In [6]:
run_client()

🔵 Client verbindet sich...
📤 Client sendet: Hello World
📥 Server empfängt: Hello World
📥 Client empfängt: Hello World *
📥 Server empfängt: STOP
🛑 Server wird beendet
